# Create Ludo Game Dataset

## Overview
This notebook generates a synthetic Ludo game dataset for training a machine learning model to predict winning combinations.

## What Was Done
- Simulated **1,000+ Ludo games** for 4 players: **Red, Green, Yellow, and Blue**
- Each player controls **4 tokens**, following standard Ludo rules:
  - All tokens begin in the home yard (**position 0**); a dice roll of **6** is required to move a token onto the board (**position 1**)
  - Active tokens advance by the dice value; overshooting position **56** causes a **bounce-back**
  - Landing on an opponent's active token **captures** it and sends it back to the home yard
  - Rolling a **6** or making a **capture** grants the player an extra turn
  - A player **wins** when all 4 of their tokens have reached **position 57** (finished)
- Simulation continues until at least **10,000 rows** of data are collected
- Each row records one turn for one player with full 4-token state information
- The final dataset is saved as `ludo_dataset.csv` in the `data file/Raw_Data/` directory

## Output
| Column | Description |
|---|---|
| `Game_ID` | Unique identifier for each simulated game |
| `Turn` | Global turn number within the game |
| `Player` | Player colour (Red, Green, Yellow, Blue) |
| `Dice_Roll` | Value rolled on the dice (1–6) |
| `Token_Moved` | Which token (1–4) was moved; 0 if no valid move was available |
| `Position_Before` | Token position before the move (0 = home, 1–56 = board, 57 = finished; NaN if no move) |
| `Position_After` | Token position after the move (same scale; NaN if no move) |
| `Tokens_Home` | Number of this player's tokens still in the home yard |
| `Tokens_Active` | Number of this player's tokens currently on the board |
| `Tokens_Finished` | Number of this player's tokens that have completed the circuit |
| `Captured_Opponent` | 1 if the move captured an opponent token, 0 otherwise |
| `Is_Winner` | 1 if this player won the game, 0 otherwise |


In [ ]:
import random
import pandas as pd
import os

HOME = 0
FINISH = 57

def simulate_ludo_optimized(target_rows=11000):
    all_records = []
    players = ['Red', 'Green', 'Yellow', 'Blue']
    game_id = 0

    while len(all_records) < target_rows:
        tokens = {p: [HOME, HOME, HOME, HOME] for p in players}
        game_buffer = []  # Store only the current game's rows
        turn_number = 0
        winner = None

        while not winner:
            for player in players:
                extra_turn = True
                while extra_turn:
                    extra_turn = False
                    dice_roll = random.randint(1, 6)
                    turn_number += 1
                    p_tokens = tokens[player]

                    # Logic: Which tokens can move?
                    can_move = [i for i, pos in enumerate(p_tokens) 
                                if pos != FINISH and (pos != HOME or dice_roll == 6)]

                    if not can_move:
                        game_buffer.append({
                            'Game_ID': game_id, 'Turn': turn_number, 'Player': player,
                            'Dice_Roll': dice_roll, 'Token_Moved': 0,
                            'Position_Before': None, 'Position_After': None,
                            'Tokens_Home': p_tokens.count(HOME),
                            'Tokens_Active': 4 - p_tokens.count(HOME) - p_tokens.count(FINISH),
                            'Tokens_Finished': p_tokens.count(FINISH),
                            'Captured_Opponent': 0, 'Is_Winner': 0
                        })
                        if dice_roll == 6: extra_turn = True
                        continue

                    # Strategy: Move active tokens first
                    active = [i for i in can_move if p_tokens[i] != HOME]
                    chosen_idx = random.choice(active if active else can_move)
                    pos_before = p_tokens[chosen_idx]

                    # Movement logic
                    if pos_before == HOME:
                        new_pos = 1
                    else:
                        new_pos_raw = pos_before + dice_roll
                        if new_pos_raw > 56:
                            new_pos = 56 - (new_pos_raw - 56) # Bounce
                        elif new_pos_raw == 56:
                            new_pos = FINISH
                        else:
                            new_pos = new_pos_raw

                    # Capture logic
                    captured = 0
                    if 1 <= new_pos <= 55:
                        for other in players:
                            if other == player: continue
                            for oidx, opos in enumerate(tokens[other]):
                                if opos == new_pos:
                                    tokens[other][oidx] = HOME
                                    captured = 1

                    p_tokens[chosen_idx] = new_pos
                    
                    game_buffer.append({
                        'Game_ID': game_id, 'Turn': turn_number, 'Player': player,
                        'Dice_Roll': dice_roll, 'Token_Moved': chosen_idx + 1,
                        'Position_Before': pos_before, 'Position_After': new_pos,
                        'Tokens_Home': p_tokens.count(HOME),
                        'Tokens_Active': 4 - p_tokens.count(HOME) - p_tokens.count(FINISH),
                        'Tokens_Finished': p_tokens.count(FINISH),
                        'Captured_Opponent': captured, 'Is_Winner': 0
                    })

                    if p_tokens.count(FINISH) == 4:
                        winner = player
                        break
                    if dice_roll == 6 or captured:
                        extra_turn = True
                if winner: break

        # Mark winner in the buffer ONLY
        for row in game_buffer:
            if row['Player'] == winner:
                row['Is_Winner'] = 1
        
        all_records.extend(game_buffer)
        game_id += 1
        print(f"Game {game_id} complete. Total rows: {len(all_records)}")

    return all_records[:target_rows] # Trim exactly to target

# Execution
data = simulate_ludo_optimized(11000)
df = pd.DataFrame(data)

# Ensure directory exists before saving
save_path = os.path.join('..', 'data file', 'Raw_Data', 'ludo_dataset.csv')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
df.to_csv(save_path, index=False)

print(f"\nSuccess! Dataset saved to {save_path}")

Dataset created with 10021 rows across 173 games, saved as 'ludo_dataset.csv'
   Game_ID  Turn  Player  Dice_Roll  Position  Is_Winner
0        0     1     Red          1         1          0
1        0     2   Green          5         5          0
2        0     3  Yellow          4         4          0
3        0     4    Blue          6         6          1
4        0     5     Red          1         2          0
5        0     6   Green          4         9          0
6        0     7  Yellow          3         7          0
7        0     8    Blue          6        12          1
8        0     9     Red          2         4          0
9        0    10   Green          2        11          0

Winner distribution:
Player
Blue      651
Green     694
Red       773
Yellow    457
dtype: int64
